# Generation from model

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

import json, torch
from tokenizers import Tokenizer
from gpt.generate import load_model, generate

RUN = "../runs/pretrain_v2"          # use "runs/pretrain_v2" if the notebook is in the repo root
model, cfg = load_model(f"{RUN}/best.pt")
tok = Tokenizer.from_file(f"{RUN}/tokenizer.json")
eot_id = json.load(open(f"{RUN}/meta.json"))["eot_id"]


def sample(prompt, max_new_tokens=200, temperature=0.8, k=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    idx = torch.tensor([[eot_id] + tok.encode(prompt).ids])      # every story starts after an end-of-text token
    out = generate(model, cfg.context_len, idx, max_new_tokens, temperature, k, g, eot_id)
    return tok.decode(out[0, 1:].tolist()), out                  # drop the leading eot when decoding


# 1) look at samples
for prompt in ["Once upon a time", "Tom and Lily"]:
    for temp in (0.7, 1.0):
        text, _ = sample(prompt, temperature=temp)
        print(f"--- {prompt!r} | temperature {temp} | top-k 50\n{text}\n")

# 2) greedy (k=1): probably repetitive, which is itself informative
text, _ = sample("Once upon a time", k=1)
print("--- greedy\n" + text + "\n")

# 3) checks
_, a = sample("Once upon a time", k=1, seed=0)
_, b = sample("Once upon a time", k=1, seed=1)
print("greedy ignores the seed:", torch.equal(a, b))

_, c = sample("Once upon a time", seed=0)
_, d = sample("Once upon a time", seed=0)
_, e = sample("Once upon a time", seed=1)
print("same seed reproduces:   ", torch.equal(c, d))
print("different seed differs: ", not torch.equal(c, e))

prompt_len = len(tok.encode("Once upon a time").ids) + 1
_, f = sample("Once upon a time", max_new_tokens=50)
print("length within limit:    ", f.shape[1] <= prompt_len + 50, "| ends with eot:", f[0, -1].item() == eot_id)

# 4) sliding window: generate far past the context length
_, g_ = sample("Once upon a time", max_new_tokens=cfg.context_len + 100, temperature=0.8)
print("generated past context_len without crashing:", g_.shape[1])

--- 'Once upon a time' | temperature 0.7 | top-k 50
Once upon a time, there was a big, ugly bug. This bug was not like the other bugs. It was very sad and lonely.

One day, the bug met a tiny bug. The bug said, "I want to be your friend. Can you make me happy?" The bug thought for a moment and said, "Yes, I will make you happy."

The bug and the bug became best friends. They played together every day. They shared food and toys. And they lived happily ever after.

--- 'Once upon a time' | temperature 1.0 | top-k 50
Once upon a time there was a little boy named Mike. He had long, curly hair and he loved to play with his scooter. One day, Mike got some clay and he rolled it around in his bedroom. He had a lot of fun bouncing it while sliding it around. 

Suddenly, Mike noticed the clay was dry. He tried to clean it but it wouldn't come off. He tried to get it out but the clay wouldn't budge. Mike was so upset and he never found the clay again. 

The next day, Mike asked his mom for help. 

In [3]:
g = torch.Generator().manual_seed(0)
idx = torch.tensor([[eot_id] + tok.encode("Once upon a time").ids])
out = generate(model, 32, idx, 150, 0.8, 50, g, -1)     # window of 32 tokens, eot_id=-1 never matches
print(out.shape)                                        # (1, T0 + 150): ran past the window without crashing
print(tok.decode(out[0, 1:].tolist())[:400])            # drifts after ~32 tokens, since the model only sees the window

torch.Size([1, 155])
Once upon a time there was a little boy named Max. He had a big, heavy box and he wanted to make something special. He asked his mom for help.

"Mommy, can I make something with this box?" Timmy asked.

His mom smiled and said, "Sure, Timmy. Let's see what we can make." They mixed flour and sugar, and soon it was ready to eat. Timmy was so happy to eat his favorite food, celery. The end.Once upon 
